# Téléchargement de données insee sur code Naf

In [45]:
df_naf = pd.read_parquet(r"C:\Users\yacin\Desktop\Analyse financière entreprises\naf_clean.parquet")


In [46]:
df_naf.head()

,siren,activitePrincipaleEtablissement
0,000325175,32.12Z
1,001807254,95.24Z
2,005410220,22.02
3,005410345,79.06
4,005410394,64.42


In [47]:
df_naf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29437547 entries, 0 to 29437546
Data columns (total 2 columns):
 #   Column                           Dtype 
---  ------                           ----- 
 0   siren                            object
 1   activitePrincipaleEtablissement  object
dtypes: object(2)
memory usage: 449.2+ MB


In [48]:
df_naf.isnull().sum()

siren                                  0
activitePrincipaleEtablissement    21390
dtype: int64

In [49]:
df_naf["activitePrincipaleEtablissement"] = df_naf["activitePrincipaleEtablissement"].fillna("NAF_INCONNU")


## Étape suivante : normaliser le code NAF

In [50]:
df_naf["naf_clean"] = df_naf["activitePrincipaleEtablissement"].str.strip()


In [51]:
# Extraire la division NAF (2 premiers chiffres)
df_naf["naf_division"] = df_naf["naf_clean"].str[:2]
df_naf.loc[df_naf["naf_clean"] == "NAF_INCONNU", "naf_division"] = None


In [52]:
# Mapping officiel INSEE (division → section)
division_to_section = {
    "01": "A", "02": "A", "03": "A",
    "05": "B", "06": "B", "07": "B", "08": "B", "09": "B",
    "10": "C", "11": "C", "12": "C", "13": "C", "14": "C", "15": "C",
    "16": "C", "17": "C", "18": "C", "19": "C", "20": "C", "21": "C",
    "22": "C", "23": "C", "24": "C", "25": "C", "26": "C", "27": "C",
    "28": "C", "29": "C", "30": "C", "31": "C", "32": "C", "33": "C",
    "35": "D",
    "36": "E", "37": "E", "38": "E", "39": "E",
    "41": "F", "42": "F", "43": "F",
    "45": "G", "46": "G", "47": "G",
    "49": "H", "50": "H", "51": "H", "52": "H", "53": "H",
    "55": "I", "56": "I",
    "58": "J", "59": "J", "60": "J", "61": "J", "62": "J", "63": "J",
    "64": "K", "65": "K", "66": "K",
    "68": "L",
    "69": "M", "70": "M", "71": "M", "72": "M", "73": "M", "74": "M", "75": "M",
    "77": "N", "78": "N", "79": "N", "80": "N", "81": "N", "82": "N",
    "84": "O",
    "85": "P",
    "86": "Q", "87": "Q", "88": "Q",
    "90": "R", "91": "R", "92": "R", "93": "R",
    "94": "S", "95": "S", "96": "S",
    "97": "T", "98": "T",
    "99": "U"
}

df_naf["naf_section"] = df_naf["naf_division"].map(division_to_section)


In [53]:
# Section → Secteur macro
section_to_macro = {
    "A": "Agriculture",
    "B": "Industries extractives",
    "C": "Industrie manufacturière",
    "D": "Énergie",
    "E": "Eau / Déchets",
    "F": "Construction",
    "G": "Commerce",
    "H": "Transport",
    "I": "Hébergement / Restauration",
    "J": "Information / Communication",
    "K": "Finance / Assurance",
    "L": "Immobilier",
    "M": "Services spécialisés",
    "N": "Services administratifs",
    "O": "Administration publique",
    "P": "Éducation",
    "Q": "Santé",
    "R": "Arts / Spectacles",
    "S": "Autres services",
    "T": "Ménages",
    "U": "Extra-territorial"
}

df_naf["secteur_macro"] = df_naf["naf_section"].map(section_to_macro)


In [55]:
df_naf["secteur_macro"] = df_naf["secteur_macro"].fillna("Secteur inconnu")


In [56]:
df_naf.head(20)

,siren,activitePrincipaleEtablissement,naf_clean,naf_division,naf_section,secteur_macro
0,000325175,32.12Z,32.12Z,32,C,Industrie manufacturière
1,001807254,95.24Z,95.24Z,95,S,Autres services
2,005410220,22.02,22.02,22,C,Industrie manufacturière
3,005410345,79.06,79.06,79,N,Services administratifs
4,005410394,64.42,64.42,64,K,Finance / Assurance
5,005410428,70.2C,70.2C,70,M,Services spécialisés
6,005410436,57.11,57.11,57,NaN,Secteur inconnu
7,005410485,64.42,64.42,64,K,Finance / Assurance
8,005410493,86.06,86.06,86,Q,Santé
9,005410501,55.4B,55.4B,55,I,Hébergement / Restauration


In [57]:
df_naf.to_parquet(r"C:\Users\yacin\Desktop\Analyse financière entreprises\naf_final.parquet", index=False)
